# TerKet Demo

This notebook shows how to use the repo, how TerKet reports backend choices, and how to run the unified benchmark script.

## Repo Quickstart

From repo root:

```powershell
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install -e .
pip install -r requirements.txt
pytest
```

Main paths:

- `src/terket/`: library code
- `benchmarks/run_benchmarks.py`: single benchmark entrypoint
- `tests/`: smoke and regression tests
- `results/`: generated CSV outputs

In [ ]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd()
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

repo_root

## Basic Amplitude Query

TerKet accepts either a native `CircuitSpec` or a Qiskit circuit. The simplest exact query asks for one transition amplitude.

In [ ]:
from terket import compute_circuit_amplitude, make_circuit

circuit = make_circuit(1, [("h", 0)])
amplitude, info = compute_circuit_amplitude(circuit, [0], [0], as_complex=True)
complex(amplitude), info

## Backend Metadata

TerKet reports structural diagnostics through `analyze_circuit()` and execution metadata through `compute_*()` helpers.

Backend names you may see in `phase3_backend`:

- `q3_free`: exact q3-free reducer handled circuit without a cubic fallback.
- `treewidth_dp` or `treewidth_dp_peeled`: exact dynamic programming on a low-treewidth cubic core.
- `q3_cover`: exact branch over a bounded cubic support cover.
- `q3_separator`: exact separator-based split of cubic core.
- `cubic_contraction_cpu`: exact contraction-style fallback on cubic residuals.
- `mixed`: different branches used different exact backends.

Optional packages can also unlock quimb and cotengra-driven planning paths used by benchmark comparisons.

In [ ]:
from terket import analyze_circuit

analysis = analyze_circuit(circuit, [0], [0])
analysis

## Qiskit Import Path

For repo users working from Qiskit, `from_qiskit()` converts into TerKet's supported basis. This is the normal bridge for imported workloads and benchmark cases.

In [ ]:
from qiskit import QuantumCircuit
from terket import from_qiskit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

spec = from_qiskit(qc)
amp, info = compute_circuit_amplitude(spec, [0, 0], [0, 0], as_complex=True)
complex(amp), info["phase3_backend"]

## Optional Backends

TerKet can run with optional native and tensor-network dependencies. This cell checks what the local environment exposes.

In [ ]:
from terket.backends import _get_quimb_tensor_module, _load_schur_native_module

{
    "native_accelerator_loaded": _load_schur_native_module() is not None,
    "quimb_available": _get_quimb_tensor_module() is not None,
}

## Unified Benchmark Script

All benchmark families now route through one script:

```text
python benchmarks/run_benchmarks.py <family> [family args]
```

Useful families:

- `head-to-head`: TerKet versus quimb amplitude timing
- `structured-showcase`: large hidden-shift showcase
- `depth-scaling`: controlled depth sweeps
- `probability-native-rcs`: probability-only RCS path
- `amplitude-post-elimination-tensor-rcs`: inspect tensor viability after elimination
- `rcs-import-strategy-probe`: compare import/transpile strategies

In [ ]:
import subprocess

result = subprocess.run(
    [
        sys.executable,
        str(repo_root / "benchmarks" / "run_benchmarks.py"),
        "head-to-head",
        "--suite",
        "smoke",
        "--repeats",
        "1",
    ],
    cwd=repo_root,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## Reading Benchmark Output

Head-to-head CSVs record both runtime and solver diagnostics:

- `cubic_obstruction` and `gauss_obstruction`: how much hard structure survives exact elimination.
- `cost_model_r`: exponent TerKet's chosen exact backend pays after reductions.
- `terket_phase3_backend`: exact backend family chosen for the hard residual.
- `quimb_over_terket_time_ratio`: quick comparison against quimb when that family includes a quimb baseline.

The `results/` directory is created on demand, so smoke runs from this notebook are safe to repeat.